In [ ]:
#!/usr/bin/env python3
"""
Gán nhãn ViSmishDS v2 cho synthetic.csv bằng OpenRouter API (model free),
tuân theo dataset_scope.md.

Output: synthetic_labeled.csv với 2 cột: label, decision_status
(đúng thứ tự dòng như synthetic.csv gốc — ghép lại với content bằng index).

Cách dùng:
    export OPENROUTER_API_KEY="sk-or-..."
    # (tuỳ chọn) export OPENROUTER_MODEL="qwen/qwen3-235b-a22b:free"
    python3 annotate.py

Resume: nếu bị ngắt giữa đường, chạy lại lệnh trên — checkpoint.jsonl giữ
các dòng đã xong, script chỉ xử lý tiếp phần còn thiếu.

QUAN TRỌNG: Đây là 'generated claim' theo §10.3/§11 của guideline, KHÔNG phải
ground truth. Cần con người review/adjudicate, đặc biệt mọi Label 1 và mọi
needs_adjudication, trước khi đưa vào release chính (§10.2.4).
"""
import csv
import json
import os
import re
import sys
import time
import urllib.request
import urllib.error

GUIDELINE_PATH = "dataset_scope.md"
INPUT_CSV = "synthetic.csv"
CHECKPOINT_PATH = "checkpoint.jsonl"
OUTPUT_CSV = "synthetic_labeled.csv"
BATCH_SIZE = 40
MAX_RETRIES = 5

# --- OpenRouter config ---
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "qwen/qwen3-235b-a22b:free")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

VALID_STATUSES = {
    "accepted_label_0",
    "accepted_label_1",
    "needs_adjudication",
    "excluded_insufficient_context",
    "excluded_invalid_record",
    "excluded_privacy_or_safety",
    "excluded_out_of_scope_malicious",
    "excluded_language_scope",
    "excluded_duplicate",
}

with open(GUIDELINE_PATH, encoding="utf-8") as f:
    GUIDELINE_TEXT = f.read()

SYSTEM_PROMPT = f"""Bạn là một annotator (LLM hỗ trợ, không phải con người) thực hiện đề xuất nhãn
sơ bộ ("generated claim") cho ViSmishDS v2, theo ĐÚNG và ĐẦY ĐỦ tài liệu chính
sách dưới đây. Bạn KHÔNG có thẩm quyền tạo ground truth — output của bạn chỉ
là gợi ý chờ con người review/adjudicate.

=== TOÀN VĂN dataset_scope.md ===
{GUIDELINE_TEXT}
=== HẾT dataset_scope.md ===

NHIỆM VỤ: Với mỗi mẫu (chỉ có trường `content`, không có provenance/category/
metadata khác — provenance được xem là `unknown` cho toàn bộ batch này), áp
dụng quy tắc ra quyết định ở mục 5 và toàn bộ guideline để xác định:

- `label`: 0, 1, hoặc null (null nếu decision_status không phải accepted_label_0/1)
- `decision_status`: MỘT trong các giá trị: accepted_label_0, accepted_label_1,
  needs_adjudication, excluded_insufficient_context, excluded_invalid_record,
  excluded_privacy_or_safety, excluded_out_of_scope_malicious,
  excluded_language_scope, excluded_duplicate

KHÔNG cần giải thích lý do — chỉ trả về hai trường trên cho mỗi mẫu.

LƯU Ý BẮT BUỘC:
- Chỉ dùng `content` làm bằng chứng (quy tắc 5.1.6); không suy đoán provenance.
- Quảng cáo cờ bạc/dịch vụ nhạy cảm không có bằng chứng lừa đảo/chiếm đoạt ->
  excluded_out_of_scope_malicious (KHÔNG ép Label 0/1).
- Tin đòi nợ đe dọa mà content không xác minh được khoản nợ thật -> accepted_label_1.
- Spam thương mại hợp pháp (không có bằng chứng lừa đảo) -> accepted_label_0.
- Nội dung chỉ có số điện thoại (không gì khác) -> accepted_label_0.
- Nội dung chỉ có URL (không gì khác) -> accepted_label_1.
- Các tín hiệu sau KHÔNG đủ một mình để gán Label 1: có URL/domain lạ, có SĐT,
  viết hoa/sai chính tả/bỏ dấu/teencode/leetspeak, tạo cảm giác cấp bách, đề
  cập ngân hàng/tiền/OTP/trúng thưởng/cơ quan nhà nước, được sinh bởi LLM,
  category cũ mang chữ "giả", model/judge dự đoán Label 1 với confidence cao.
- Nếu hai cách diễn giải (hợp lệ vs lừa đảo) cạnh tranh ngang nhau -> needs_adjudication.
- Nếu nhãn chỉ xác định được nhờ ngữ cảnh ngoài bản ghi (hội thoại thiếu, v.v.)
  -> excluded_insufficient_context.
- Nội dung rỗng/hỏng encoding/ký tự không diễn giải được -> excluded_invalid_record.
- Hơn 50% nội dung là tiếng Anh (không tính brandname/viết tắt thông dụng) ->
  excluded_language_scope.
- Chứa thông tin cá nhân nhạy cảm không thể khử định danh an toàn (SĐT cá nhân
  thật kèm tên thật, CCCD, số tài khoản cụ thể của một cá nhân có thể nhận diện,
  v.v., khi việc công bố gây hại) -> excluded_privacy_or_safety. (Số điện thoại
  hotline/shortcode/brandname không tính vào đây.)
- Đây là dữ liệu synthetic: nếu nội dung có lỗi logic nghiêm trọng, mâu thuẫn,
  placeholder hỏng (ví dụ "{{tên}}" chưa điền, số liệu vô nghĩa làm mẫu không
  còn diễn giải được) -> excluded_invalid_record. Lỗi chính tả/teencode thông
  thường KHÔNG tính vào đây.
- KHÔNG suy diễn nhóm đối tượng nhân khẩu học; điều này không ảnh hưởng label
  ở đây vì không có cột metadata nhóm đối tượng trong batch này — bỏ qua mục 9.

ĐỊNH DẠNG TRẢ VỀ: CHỈ một JSON array thuần (không markdown, không code fence,
không giải thích thêm), mỗi phần tử tương ứng đúng thứ tự mẫu đầu vào:
[{{"label": 0, "decision_status": "accepted_label_0"}}, ...]
Số phần tử trong array PHẢI bằng đúng số mẫu trong batch đầu vào."""


def call_openrouter(batch_contents, attempt=0):
    user_msg = "Gán nhãn cho các mẫu sau (mỗi mẫu đánh số):\n\n"
    for i, c in enumerate(batch_contents):
        user_msg += f"[{i}] {c}\n\n"
    user_msg += f"\nTrả về JSON array với đúng {len(batch_contents)} phần tử, theo đúng thứ tự trên."

    body = json.dumps({
        "model": OPENROUTER_MODEL,
        "max_tokens": 3500,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    }).encode("utf-8")

    req = urllib.request.Request(
        OPENROUTER_URL,
        data=body,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8")
        if e.code == 429 or e.code >= 500:
            if attempt < MAX_RETRIES:
                wait = 2 ** attempt
                print(f"  HTTP {e.code}, retry in {wait}s... ({err_body[:200]})", file=sys.stderr)
                time.sleep(wait)
                return call_openrouter(batch_contents, attempt + 1)
        raise RuntimeError(f"HTTP {e.code}: {err_body}")

    try:
        text = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError):
        raise RuntimeError(f"Unexpected response shape: {json.dumps(data)[:500]}")
    return text.strip()


def parse_response(text, expected_n):
    # strip potential code fences just in case
    cleaned = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        # try to find the first [ ... ] block
        m = re.search(r"\[.*\]", cleaned, re.DOTALL)
        if not m:
            raise
        parsed = json.loads(m.group(0))

    if not isinstance(parsed, list):
        raise ValueError("Response is not a JSON array")
    if len(parsed) != expected_n:
        raise ValueError(f"Expected {expected_n} items, got {len(parsed)}")

    out = []
    for item in parsed:
        status = item.get("decision_status")
        if status not in VALID_STATUSES:
            raise ValueError(f"Invalid decision_status: {status}")
        label = item.get("label")
        if status == "accepted_label_0":
            label = 0
        elif status == "accepted_label_1":
            label = 1
        else:
            label = None
        out.append({
            "label": label,
            "decision_status": status,
        })
    return out


def load_checkpoint():
    done = {}
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                done[rec["idx"]] = rec["result"]
    return done


def append_checkpoint(idx, result):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({"idx": idx, "result": result}, ensure_ascii=False) + "\n")


def main():
    with open(INPUT_CSV, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    n = len(rows)
    print(f"Tổng số mẫu: {n}")

    done = load_checkpoint()
    print(f"Đã có checkpoint cho {len(done)} mẫu")

    indices_to_process = [i for i in range(n) if i not in done]
    print(f"Còn cần xử lý: {len(indices_to_process)} mẫu")

    for batch_start in range(0, len(indices_to_process), BATCH_SIZE):
        batch_idx = indices_to_process[batch_start: batch_start + BATCH_SIZE]
        if not batch_idx:
            continue
        batch_contents = [rows[i]["content"] for i in batch_idx]

        success = False
        for attempt in range(MAX_RETRIES):
            try:
                raw = call_openrouter(batch_contents)
                results = parse_response(raw, len(batch_idx))
                success = True
                break
            except Exception as e:
                print(f"  Batch {batch_idx[0]}-{batch_idx[-1]} lỗi (attempt {attempt+1}): {e}", file=sys.stderr)
                time.sleep(1 + attempt)

        if not success:
            print(f"BỎ QUA batch {batch_idx[0]}-{batch_idx[-1]} sau {MAX_RETRIES} lần thử, sẽ retry ở lần chạy sau.", file=sys.stderr)
            continue

        for i, res in zip(batch_idx, results):
            append_checkpoint(i, res)

        processed_total = len(load_checkpoint())
        print(f"Done {processed_total}/{n}")

    # Final assembly
    done = load_checkpoint()
    missing = [i for i in range(n) if i not in done]
    if missing:
        print(f"CẢNH BÁO: còn {len(missing)} mẫu chưa được gán (chạy lại script để resume). Vẫn xuất các mẫu đã có.")

    with open(OUTPUT_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["label", "decision_status"])
        writer.writeheader()
        for i in range(n):
            if i in done:
                r = done[i]
                writer.writerow({
                    "label": "" if r["label"] is None else r["label"],
                    "decision_status": r["decision_status"],
                })
            else:
                writer.writerow({
                    "label": "",
                    "decision_status": "PENDING_NOT_PROCESSED",
                })

    print(f"Đã xuất {OUTPUT_CSV}")


if __name__ == "__main__":
    main()